In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
df = pd.read_csv("Task 3 and 4_Loan_Data.csv")
df.head()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
0,8153374,0,5221.545193,3915.471226,78039.38546,5,605,0
1,7442532,5,1958.928726,8228.752520,26648.43525,2,572,1
2,2256073,0,3363.009259,2027.830850,65866.71246,4,602,0
3,4885975,0,4766.648001,2501.730397,74356.88347,5,612,0
4,4700614,1,1345.827718,1768.826187,23448.32631,6,631,0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               10000 non-null  int64  
 1   credit_lines_outstanding  10000 non-null  int64  
 2   loan_amt_outstanding      10000 non-null  float64
 3   total_debt_outstanding    10000 non-null  float64
 4   income                    10000 non-null  float64
 5   years_employed            10000 non-null  int64  
 6   fico_score                10000 non-null  int64  
 7   default                   10000 non-null  int64  
dtypes: float64(3), int64(5)
memory usage: 625.1 KB


In [5]:
df.describe()

,customer_id,credit_lines_outstanding,loan_amt_outstanding,total_debt_outstanding,income,years_employed,fico_score,default
count,1.000000e+04,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,4.974577e+06,1.461200,4159.677034,8718.916797,70039.901401,4.552800,637.557700,0.185100
std,2.293890e+06,1.743846,1421.399078,6627.164762,20072.214143,1.566862,60.657906,0.388398
min,1.000324e+06,0.000000,46.783973,31.652732,1000.000000,0.000000,408.000000,0.000000
25%,2.977661e+06,0.000000,3154.235371,4199.836020,56539.867903,3.000000,597.000000,0.000000
50%,4.989502e+06,1.000000,4052.377228,6732.407217,70085.826330,5.000000,638.000000,0.000000
75%,6.967210e+06,2.000000,5052.898103,11272.263740,83429.166133,6.000000,679.000000,0.000000
max,8.999789e+06,5.000000,10750.677810,43688.784100,148412.180500,10.000000,850.000000,1.000000


In [8]:
fico_score = df["fico_score"].values
default = df["default"].values

In [10]:
def build_score_table(fico_score, default):
    """
    Aggregate raw records into a table indexed by unique FICO score,
    with count of borrowers (n) and count of defaults (k) at each score.
    """

    df_temp = pd.DataFrame({'fico':fico_score, 'default': default})
    grouped = df_temp.groupby('fico').agg(
        n = ('default', 'count'),
        k = ('default', 'sum')
    ).reset_index()
    grouped = grouped.sort_values('fico').reset_index(drop = True)
    return grouped

score_table = build_score_table(fico_score, default)

In [26]:
def bucket_log_likelihood(n,k):
    """
    Log-likelihood contribution of one bucket given n borrowers and k defaults,
    under the assumption defaults ~ Binomial(n, p), p = k/n (MLE).
    
    LL = k*log(p) + (n-k)*log(1-p)
    """
    if n == 0:
        return 0
    p = k/n
    if p == 1 or p == 0:
        return 0
    return k*np.log(p) + (n-k)*np.log(1 - p)

In [27]:
def compute_prefix_table(score_table):
    """
    Precompute cumulative n and k so that stats for any bucket [i, j)
    can be retrieved in O(1) instead of re-summing every time.
    """

    n_arr = score_table['n'].values
    k_arr = score_table['k'].values

    prefix_n = np.concatenate(([0], np.cumsum(n_arr)))#joins two arrays - concatenate
    prefix_k = np.concatenate(([0], np.cumsum(k_arr)))
    return prefix_n, prefix_k

def bucket_stats(prefix_n, prefix_k, i, j):
    """ 
    retuen (n, k) for the bucket covering score_table rows [i,j)
    """
    n = prefix_n[j] - prefix_n[i]
    k = prefix_k[j] - prefix_k[i]
    return n, k

In [43]:
def optimal_buckets_loglikelihood(score_table, num_buckets):
    """
    Find bucket boundaries over unique FICO scores that maximize total
    log-likelihood, using DP.
    
    State: dp[b][i] = best total log-likelihood using b buckets to cover
                       the first i unique scores (indices 0..i-1).
    Transition: dp[b][i] = max over j < i of dp[b-1][j] + LL(bucket [j, i))
    """
    U = len(score_table)  # number of unique FICO scores
    prefix_n, prefix_k = compute_prefix_table(score_table)
    
    # Precompute LL for every possible bucket [i, j) once, reused across b
    # ll_cache[i][j] = log-likelihood of bucket spanning indices i..j-1
    ll_cache = np.full((U + 1, U + 1), -np.inf)
    for i in range(U):
        for j in range(i + 1, U + 1):
            n, k = bucket_stats(prefix_n, prefix_k, i, j)
            ll_cache[i][j] = bucket_log_likelihood(n, k)
    
    # dp[b][i]: max total LL using b buckets over first i scores
    dp = np.full((num_buckets + 1, U + 1), -np.inf)
    dp[0][0] = 0  # 0 buckets, 0 scores covered -> LL = 0 (base case)
    
    # split[b][i]: the boundary j that achieves dp[b][i], for backtracking
    split = np.zeros((num_buckets + 1, U + 1), dtype=int)
    
    for b in range(1, num_buckets + 1):
        for i in range(1, U + 1):
            # try every possible start point j for the b-th bucket: [j, i)
            for j in range(b - 1, i):  # need at least b-1 scores for prior buckets
                if dp[b - 1][j] == -np.inf:
                    continue
                candidate = dp[b - 1][j] + ll_cache[j][i]
                if candidate > dp[b][i]:
                    dp[b][i] = candidate
                    split[b][i] = j
    
    return dp, split, score_table

In [44]:
def exact_boundaries(dp, split, score_table, num_buckets):
    """
    Walk backward through the split table to recover the actual
    FICO score boundaries.
    """

    u = len(score_table)
    boundaries_idx = []
    i = u
    b = num_buckets

    while b>0:
        j = split[b][i]
        boundaries_idx.append(j)
        i = j
        b -=1

    boundaries_idx.append(0)
    boundaries_idx = sorted(set(boundaries_idx))

    #convert index boyundaries into actual FICO score ut points

    fico_values = score_table['fico'].values
    score_boundaries = [fico_values[idx] for idx in boundaries_idx[:-1]]
    score_boundaries.append(fico_values[-1] + 1) #upper bound, exclusive

    return score_boundaries

best_ll = None 

In [45]:
def assign_ratings(fico_score, boundaries):
    """
    Map a single FICO score to a rating using the boundaries.
    Rating 0 = best (highest FICO), rating (num_buckets-1) = worst.
    """
    num_buckets = len(boundaries) - 1
    for i in range (num_buckets):
        if boundaries[i] <= fico_score < boundaries[i + 1]:
            
            
         # Buckets are ordered low-to-high FICO; we want low rating = high FICO
         # so we reverse the index   
            return num_buckets - 1 - i
    return None #score out of range

def build_rating_map(boundaries):
    num_buckets = len(boundaries) - 1
    rating_map = []
    for i in range(num_buckets):
        rating_map.append({
            'rating': num_buckets - 1 -i,
            'fico_min': boundaries[i],
            'fico_max': boundaries[i+1] - 1
        })
    return pd.DataFrame(rating_map).sort_values('rating').reset_index(drop=True)


In [46]:
def fit_fico_buckets(fico_scores, defaults, num_buckets):
    """
    Full pipeline: raw data -> optimal boundaries -> rating map.
    """
    score_table = build_score_table(fico_scores, defaults)
    dp, split, score_table = optimal_buckets_loglikelihood(score_table, num_buckets)
    boundaries = exact_boundaries(dp, split, score_table, num_buckets)
    rating_map = build_rating_map(boundaries)
    best_total_ll= dp[num_buckets][len(score_table)]

    return rating_map, boundaries, best_total_ll

def apply_rating_map(fico_scores, boundaries):
    return np.array([assign_ratings(s, boundaries) for s in fico_scores])

In [47]:
rating_map, boundaries, best_ll = fit_fico_buckets(fico_score, default, 10)
print("Boundaries:", boundaries)
print("Best LL:", best_ll)
print(rating_map)

ratings = apply_rating_map(fico_score, boundaries)
check = pd.DataFrame({'fico': fico_score, 'default': default, 'rating': ratings})
print(check.groupby('rating')['default'].mean().sort_index())

Boundaries: [np.int64(408), np.int64(521), np.int64(553), np.int64(581), np.int64(612), np.int64(650), np.int64(697), np.int64(733), np.int64(753), np.int64(851)]
Best LL: -4217.824477362505
   rating  fico_min  fico_max
0       0       753       850
1       1       733       752
2       2       697       732
3       3       650       696
4       4       612       649
5       5       581       611
6       6       553       580
7       7       521       552
8       8       408       520
rating
0    0.032000
1    0.016502
2    0.057971
3    0.098122
4    0.163083
5    0.244074
6    0.336992
7    0.461694
8    0.661130
Name: default, dtype: float64
